# Lesson 10 — compaction & cleanup · เก็บกวาดสิ่งที่ append ทิ้งไว้

ทุกบทที่ผ่านมาบอกว่า Lance ไม่เขียนทับ มีแต่เขียนเพิ่ม
บทนี้ดูว่าเพิ่มไปเรื่อย ๆ แล้วเกิดอะไร fragment เล็ก ๆ กองเต็ม `_deletions/` โต version พอก
แล้วสั่งเก็บกวาดสองขั้น `compact_files()` รวม fragment · `cleanup_old_versions()` ลบ version เก่า
สองคำสั่งนี้แยกกัน คนละหน้าที่ ทำทีละอย่างแล้วนับ

In [1]:
%pip install -q lancedb pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, urllib.request, pathlib, warnings
warnings.filterwarnings("ignore")   # compact_files / cleanup_old_versions warn "deprecated" — see below
if not pathlib.Path("../data/lesson_data.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Soul-Brews-Studio/lancedb-oracle/main/lessons/data/lesson_data.py", "lesson_data.py")
sys.path.insert(0, "../data")
from lesson_data import load

from pathlib import Path
import lancedb

db = lancedb.connect("./data")
tbl = db.create_table("posts", data=load("nat_posts.jsonl"), mode="overwrite")

def state(label):
    root = Path("data/posts.lance")
    frags = len(list((root / "data").glob("*.lance")))
    dels = len(list((root / "_deletions").glob("*"))) if (root / "_deletions").exists() else 0
    vers = len(list((root / "_versions").glob("*.manifest")))
    size = sum(f.stat().st_size for f in root.rglob("*") if f.is_file())
    print(f"{label:<22} rows={tbl.count_rows():<3} fragments={frags:<3} deletions={dels:<3} versions={vers:<3} bytes={size}")

state("start")

start                  rows=11  fragments=1   deletions=0   versions=1   bytes=5885


[2026-09-10T11:36:42Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/10-compaction/data/posts.lance, it will be created


**กองให้ดู** — update 5 ครั้ง add 3 ครั้ง delete 2 ครั้ง รวม 10 การเขียน
ทุก `add` = fragment ใหม่ ทุก `update`/`delete` = ไฟล์ใน `_deletions/` ทุกอย่าง = version ใหม่
ตารางมีแค่ 11 แถว แต่บน disk เหมือนตารางที่ถูกเขียน 11 ครั้ง

In [3]:
for i in range(5):
    tbl.update(where=f"id = 'p0{i + 1}'", values={"topic": "memory"})
for i in range(3):
    tbl.add([{"id": f"x{i}", "date": "2026-09-10", "topic": "agents", "vector": [0.0, 1.0, 0.0], "text": f"extra {i}"}])
tbl.delete("id = 'x0'")
tbl.delete("id = 'x1'")
state("after 10 writes")

after 10 writes        rows=12  fragments=9   deletions=5   versions=11  bytes=34617


**ขั้นที่ 1 — `compact_files()`** รวม fragment เล็ก ๆ เป็นก้อนเดียว
แถวที่ถูก mark ลบไว้ ถูกทิ้งจริงตอนเขียนก้อนใหม่ manifest ล่าสุดชี้ไป fragment เดียว
แต่บน disk ยังนับได้ 10 fragment 5 deletion file เพราะไฟล์เก่าไม่ถูกลบ version เพิ่มอีกสอง bytes ยิ่งโต
ตั้งแต่ 0.21 คำสั่งนี้ถูกทำเครื่องหมาย deprecated ให้ใช้ `optimize()` แทน ยังเรียกได้ เอาไว้ดูทีละขั้น

In [4]:
stats = tbl.compact_files()
print(stats)
state("after compact_files")

CompactionMetrics(fragments_removed=7, fragments_added=1, files_removed=8, files_added=1)
after compact_files    rows=12  fragments=10  deletions=5   versions=13  bytes=42475


**ขั้นที่ 2 — `cleanup_old_versions()`** ลบ manifest กับ fragment ที่ไม่มี version ไหนชี้ถึงแล้ว
`older_than=timedelta(0)` = ลบทุก version ที่ไม่ใช่ล่าสุด ปกติใส่ 7 วันเผื่อ reader ที่ค้างอยู่
ตรงนี้แหละที่ Nothing is Deleted หยุด เพราะเราสั่งเอง (deprecated ตั้งแต่ 0.21 เช่นกัน)
ลำดับสำคัญ compact ก่อน cleanup ทีหลัง สลับกันแล้ว cleanup ไม่มีอะไรให้ลบ
ผลคือกลับมา fragment 1 · deletion 0 · version 1 เท่าตอนเริ่ม แต่ข้อมูลเป็นชุดใหม่

In [5]:
from datetime import timedelta
stats = tbl.cleanup_old_versions(older_than=timedelta(0))
print(stats)
state("after cleanup")

CleanupStats { bytes_removed: 35622, old_versions: 12, data_files_removed: 9, transaction_files_removed: 12, index_files_removed: 0, deletion_files_removed: 5 }
after cleanup          rows=12  fragments=1   deletions=0   versions=1   bytes=6853


**index ก็ค้างได้** — สร้าง FTS index แล้ว `add` แถวใหม่
ยังค้นเจอ เพราะ Lance สแกน fragment ที่ยังไม่มี index ให้แบบ brute force
แต่ `index_stats` บอกความจริง `num_unindexed_rows = 1` แถวนั้นอยู่นอก index ค้นช้ากว่า
`optimize()` = compact + cleanup + เติม index ในคำสั่งเดียว นี่คือคำสั่งที่ 0.38 อยากให้ใช้ ตั้ง schedule วันละครั้ง

In [6]:
from lancedb.index import FTS
tbl.create_index("text", config=FTS(base_tokenizer="icu"))
tbl.add([{"id": "x9", "date": "2026-09-10", "topic": "agents", "vector": [0.0, 1.0, 0.0], "text": "oracle ตัวใหม่ตื่นแล้ว"}])
s = tbl.index_stats("text_idx")
print("before optimize: found", [h["id"] for h in tbl.search("oracle", query_type="fts").limit(3).to_list()],
      "| indexed", s.num_indexed_rows, "unindexed", s.num_unindexed_rows)
tbl.optimize(cleanup_older_than=timedelta(0))
s = tbl.index_stats("text_idx")
print("after optimize:  found", [h["id"] for h in tbl.search("oracle", query_type="fts").limit(3).to_list()],
      "| indexed", s.num_indexed_rows, "unindexed", s.num_unindexed_rows)
state("after optimize")

before optimize: found ['x9', 'p05', 'p01'] | indexed 12 unindexed 1
after optimize:  found ['x9', 'p05', 'p01'] | indexed 13 unindexed 0
after optimize         rows=13  fragments=2   deletions=0   versions=2   bytes=31866


ตารางที่ agent เขียนทุกวันแล้วไม่เคย optimize จะโตแบบ "after 10 writes" ไปเรื่อย ๆ
ใน fleet มีตาราง 835 MB ที่น่าจะไม่เคยผ่านคำสั่งนี้เลย
`optimize(cleanup_older_than=timedelta(days=7))` วันละครั้ง คือสิ่งที่บทนี้อยากให้จำ